# Sentence-level Retrieval

Giữ **một vector cho mỗi câu** trong bài báo.

Nếu bài báo có `N` câu:

```text
sentence 1 -> [768]
sentence 2 -> [768]
...
sentence N -> [768]

=> document_sentence_embeddings: [N, 768]
```

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

/Users/thangtran/Workplace/master_s_degree/information_retrieval/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "vinai/phobert-base"

CACHE_DIR = "../../data/models/vinai-phobert"

phoBert = AutoModel.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)

phoBert.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51605.45it/s]
[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(64001, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(258, 768, padding_idx=1)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=Tru

In [3]:
segmented_sentences = [
    'Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .', 
    'Chiều về , vé cũng được áp_dụng mức giá khuyến_mại 19.000 đồng , nhưng sau khi cộng thuế , phí , tổng tiền chị Loan phải trả khoảng 2,4 triệu đồng .', 
    'Theo chị , các khoản phí tại sân_bay Singapore cao hơn chiều bay từ Việt_Nam nên dù cùng giá vé niêm_yết , số tiền thực trả vẫn chênh_lệch đáng_kể .', 
    'Tính cả hai chiều , chuyến đi Singapore của chị hết hơn 4 triệu đồng , giảm khoảng một_nửa so với cùng kỳ năm_ngoái .', 
    'Sau Covid-19 , các đường_bay quốc_tế mất nhiều thời_gian để phục_hồi , trong khi nguồn cung chưa trở_lại như trước khiến giá luôn ở mức cao , nhất_là vào mùa du_lịch .', 
    'Năm nay , nguồn cung tăng nhanh hơn , kéo_theo cạnh_tranh giữa các hãng và tạo thêm dư_địa giảm_giá .', 
    'Mức giá chị Loan mua không phải trường_hợp cá_biệt .', 
    'Khảo_sát các đường_bay từ TP HCM đi Singapore và Thái_Lan cho thấy mức giá khuyến_mại 19.000-90.000 đồng , chưa gồm thuế , phí chiếm đa_số các chặng bay trong tháng 8 và 9 .', 
    'Sau khi cộng các khoản này , vé TP HCM - Singapore từ hơn 1,6 triệu đồng một_chiều , còn chặng TP HCM - Bangkok chưa đến 1,9 triệu đồng .', 
    'Vé của một_số hãng hàng_không nước_ngoài trên cùng_đường bay hiện cao hơn khoảng 2-3 lần so với các hãng Việt_Nam .', 
    'Từ Hà_Nội đi Singapore và Thái_Lan , giá vé của các hãng trong nước dao_động 2,6-3 triệu đồng một_chiều , đã gồm thuế , phí .', 
    'Một_số ngày trong tháng 8 , mức thấp nhất còn hơn 2 triệu đồng .', 
    'Với đường_bay TP HCM - Jakarta , giá cũng giảm nhưng mặt_bằng vẫn cao hơn Singapore và Thái_Lan .', 
    'Nếu trước_đây vé khứ_hồi thường ở mức 7-10 triệu đồng , hiện giá thấp nhất khoảng 6,3 triệu đồng , đã gồm thuế , phí , tương_đương hơn 3 triệu đồng mỗi chiều .', 
    'Mức giá cao hơn một phần do quãng đường_bay xa hơn .', 
    'Các đường_bay từ Hà_Nội và TP HCM tới châu_Âu , Đông_Bắc_Á cũng giảm khoảng 10-15% so với trước .', 
    'Giá đi xuống trong bối_cảnh nguồn cung hàng_không Việt_Nam tăng .', 
    'Theo dữ_liệu dự_báo của Công_ty cung_cấp dữ_liệu hàng không OAG ( Anh ) , Việt_Nam có khoảng 7,3 triệu ghế cung_ứng trong tháng 8 , tăng 10% so với cùng kỳ năm_ngoái và đứng thứ hai Đông_Nam_Á , sau Indonesia .', 
    'Trong khi tổng năng_lực khai_thác của thị_trường hàng_không Đông_Nam_Á tháng 8 chỉ tăng 0,8% so với cùng kỳ , nguồn cung của Việt_Nam tăng tới 10% .', 
    'Trong đó , Vietnam_Airlines có khoảng 2,8 triệu ghế , tăng 8,2% , trong khi Vietjet khoảng 2,24 triệu ghế .', 
    'Nguồn cung trên các đường_bay quốc_tế cũng được tăng_cường .', 
    'Vietjet_Air cho biết , nâng tần_suất TP HCM - Kuala_Lumpur lên 7 chuyến mỗi tuần trong mùa cao_điểm , đồng_thời mở đường_bay TP HCM - Colombo từ ngày 18/8 .', 
    'Hãng cũng chuẩn_bị khai_thác các đường_bay Hà_Nội - Almaty và Hà_Nội - Praha từ tháng 10 .', 
    'Ông Hồng_Thanh , chủ một đại_lý vé máy_bay tại TP HCM , cho biết nguồn cung tăng và cạnh_tranh giữa các hãng là nguyên_nhân quan_trọng khiến giá vé quốc_tế hạ nhiệt .', 
    'Các hãng phải tăng khuyến_mại , kích_cầu trong bối_cảnh sức_mua chưa phục_hồi như kỳ_vọng .', 
    'Chi_phí nhiên_liệu cũng thuận_lợi hơn cho các hãng .', 
    'Từ ngày 1/7 , Chính_phủ tiếp_tục kéo_dài thời_hạn áp_dụng thuế nhập_khẩu ưu_đãi , thuế bảo_vệ môi_trường và thuế_giá_trị gia_tăng với xăng_dầu , nhiên_liệu bay đến hết ngày 30/9/2026 , giúp giảm một phần chi_phí đầu_vào của các hãng hàng_không .', 
    'Về nhu_cầu , thị_trường khách quốc_tế đến Việt_Nam tăng mạnh .', 
    'Bảy tháng đầu năm , Việt_Nam đón gần 14 triệu lượt khách quốc_tế , tăng gần 14% so với cùng kỳ năm_ngoái .', 
    'Riêng tháng 7 , lượng khách đạt khoảng 1,67 triệu lượt , trong đó đường_hàng không chiếm gần 83% .', 
    'Nhu_cầu đi_lại quốc_tế tăng trong khi nguồn cung được bổ_sung khiến các hãng phải cạnh_tranh mạnh hơn để thu_hút khách .', 
    'Đây cũng là một trong những yếu_tố kéo mặt_bằng giá xuống trong mùa hè năm nay .', 
    'Không_chỉ quốc_tế , trước đó các hãng cũng liên_tục kích_cầu trên thị_trường nội_địa ngay giữa cao_điểm hè .', 
    'Nhiều chương_trình đưa giá vé một_số chặng về mức 0 đồng hoặc vài chục nghìn đồng , chưa gồm thuế , phí .'
]

print(f"Array length: {len(segmented_sentences)}")

Array length: 34


In [4]:
def mean_pooling(
    token_embeddings: torch.Tensor,
    pooling_mask: torch.Tensor
) -> torch.Tensor:
    """
    Mean Pooling theo chiều token.

    Input:
        token_embeddings: [batch_size, num_tokens, 768]
        pooling_mask:     [batch_size, num_tokens]

    Output:
        sentence_embeddings: [batch_size, 768]
    """
    mask = pooling_mask.unsqueeze(-1).float()

    sum_embeddings = torch.sum(
        token_embeddings * mask,
        dim=1
    )

    token_count = torch.sum(
        mask,
        dim=1
    ).clamp(min=1e-9)

    return sum_embeddings / token_count

In [5]:
def encode_sentences(
    sentences: list[str],
    max_length: int = 256
) -> torch.Tensor:
    """
    Encode nhiều câu cùng lúc bằng PhoBERT.

    Output:
        Tensor [num_sentences, 768]
    """
    encoded = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        output = phoBert(**encoded)

    token_embeddings = output.last_hidden_state

    # Tạo mask cho special tokens: <s>, </s>, <pad>, ...
    special_tokens_mask = torch.tensor([
        tokenizer.get_special_tokens_mask(
            ids,
            already_has_special_tokens=True
        )
        for ids in encoded["input_ids"].tolist()
    ])

    # attention_mask loại padding;
    # (1 - special_tokens_mask) loại special tokens.
    content_mask = (
        encoded["attention_mask"]
        * (1 - special_tokens_mask)
    )

    sentence_embeddings = mean_pooling(
        token_embeddings,
        content_mask
    )

    return sentence_embeddings


In [6]:
document_sentence_embeddings = encode_sentences(
    segmented_sentences
)

print(f"Shape: {document_sentence_embeddings.shape}")

Shape: torch.Size([34, 768])


Ví dụ 34 câu:
torch.Size([34, 768])

In [7]:
sentence_records = []

for sentence_id, (sentence, embedding) in enumerate(
    zip(segmented_sentences, document_sentence_embeddings)
):
    sentence_records.append({
        "sentence_id": sentence_id,
        "text": sentence,
        "embedding": embedding
    })

print(f"Number of records: {len(sentence_records)}")
print(sentence_records[0]["sentence_id"])
print(sentence_records[0]["text"])
print(sentence_records[0]["embedding"].shape)


Number of records: 34
0
Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .
torch.Size([768])


In [8]:
print("Sentence:")
print(sentence_records[0]["text"])

print("\nEmbedding shape:")
print(sentence_records[0]["embedding"].shape)

print("\nFirst 10 dimensions:")
print(sentence_records[0]["embedding"][:10])


Sentence:
Đặt vé từ TP HCM đi Singapore để công_tác , chị Hoàng_Loan , ở phường Xuân_Hoà , bất_ngờ vì mức giá lần đầu mua được kể từ sau đại_dịch " Năm_ngoái , chặng TP HCM - Singapore có lúc lên tới 3,2 triệu đồng một_chiều , còn năm nay tôi chỉ trả hơn 1,6 triệu đồng , đã gồm thuế , phí " , chị nói .

Embedding shape:
torch.Size([768])

First 10 dimensions:
tensor([ 0.0262,  0.0773, -0.1331, -0.2164,  0.1964, -0.4092, -0.2246, -0.2143,
         0.0801,  0.2767])


## 7. Lưu output để notebook khác sử dụng

In [9]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/embeddings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

V1_OUTPUT_PATH = OUTPUT_DIR / "v1_sentence_level_retrieval_embeddings.pt"

v1_output = {
    "sentences": segmented_sentences,
    "embeddings": document_sentence_embeddings.detach().cpu()
}

torch.save(v1_output, V1_OUTPUT_PATH)

print(f"Saved: {V1_OUTPUT_PATH}")
print(f"Embeddings shape: {v1_output['embeddings'].shape}")

Saved: ../../data/embeddings/v1_sentence_level_retrieval_embeddings.pt
Embeddings shape: torch.Size([34, 768])
